<a href="https://colab.research.google.com/github/e-saidha/skyhack_querykings/blob/main/Copy_of_skyhack_all_deliverables_official.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
flights = pd.read_csv('/content/Flight Level Data.csv')
pnr_flight = pd.read_csv('/content/PNR+Flight+Level+Data.csv')
pnr_rem = pd.read_csv('/content/PNR Remark Level Data.csv')
bag_level = pd.read_csv('/content/Bag+Level+Data.csv')
airports = pd.read_csv('/content/Airports Data.csv')
airports = airports.drop_duplicates()

In [ ]:
import duckdb, pandas as pd, numpy as np
con = duckdb.connect()


con.register("flights", flights)
con.register("pnr_flight", pnr_flight)
con.register("pnr_rem", pnr_rem)
con.register("bag_level", bag_level)
con.register("airports", airports)


In [ ]:

con.execute("""
CREATE OR REPLACE VIEW v_flights_ord AS
SELECT *
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY company_id, flight_number, scheduled_departure_date_local,
                   scheduled_departure_station_code, scheduled_arrival_station_code
      ORDER BY COALESCE(actual_departure_datetime_local, scheduled_departure_datetime_local) DESC
    ) AS rn
  FROM flights
  WHERE scheduled_departure_station_code = 'ORD'
) t
WHERE rn = 1
""")
n_flights = con.execute("SELECT COUNT(*) FROM v_flights_ord").fetchone()[0]



In [ ]:

con.execute("""
CREATE OR REPLACE VIEW v_pnr_per_booking AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  record_locator,
  -- latest snapshot per booking (handle updates)
  MAX(COALESCE(total_pax,0))                                              AS pnr_pax,
  MAX(COALESCE(lap_child_count,0))                                        AS pnr_lap_childs,
  MAX(COALESCE(CAST(basic_economy_ind AS BIGINT),0))                      AS pnr_basic_econ,
  MAX(
    CASE
      WHEN UPPER(TRIM(CAST(is_stroller_user AS VARCHAR))) IN ('Y','YES','TRUE','1') THEN 1
      ELSE 0
    END
  )                                                                        AS pnr_stroller_user
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY company_id, flight_number, scheduled_departure_date_local,
                   scheduled_departure_station_code, scheduled_arrival_station_code,
                   record_locator
      ORDER BY pnr_creation_date DESC
    ) AS rn
  FROM pnr_flight
) q
WHERE rn = 1
GROUP BY 1,2,3,4,5,6
""")


con.execute("""
CREATE OR REPLACE VIEW v_pnr_agg AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  SUM(pnr_pax)           AS pax_total,
  SUM(pnr_lap_childs)    AS lap_childs,
  SUM(pnr_basic_econ)    AS basic_econ,
  SUM(pnr_stroller_user) AS stroller_users,
  COUNT(*)               AS distinct_pnrs
FROM v_pnr_per_booking
GROUP BY 1,2,3,4,5
""")


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW v_bags_agg AS
SELECT
  company_id, flight_number, scheduled_departure_date_local,
  scheduled_departure_station_code, scheduled_arrival_station_code,
  -- origin == checked
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(bag_type))='ORIGIN' THEN bag_tag_unique_number END)        AS checked_bags,
  -- transfer includes both 'transfer' and 'hot transfer'
  COUNT(DISTINCT CASE WHEN UPPER(TRIM(bag_type)) IN ('TRANSFER','HOT TRANSFER')
                      THEN bag_tag_unique_number END)                                           AS transfer_bags
FROM bag_level
GROUP BY 1,2,3,4,5
""")


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW v_ssr_agg AS
SELECT
  p.company_id, p.flight_number, p.scheduled_departure_date_local,
  p.scheduled_departure_station_code, p.scheduled_arrival_station_code,
  COUNT(*) AS ssr_count
FROM pnr_rem r
JOIN v_pnr_per_booking p
  ON r.record_locator = p.record_locator
 AND r.flight_number  = p.flight_number
GROUP BY 1,2,3,4,5
""")


In [ ]:
df_master = con.execute("""
WITH base AS (
  SELECT
    f.*,
    (f.scheduled_ground_time_minutes - f.minimum_turn_minutes) AS slack_mins,
    EXTRACT(hour FROM CAST(f.scheduled_departure_datetime_local AS TIMESTAMP)) AS dep_hour,
    EXTRACT(dow  FROM CAST(f.scheduled_departure_datetime_local AS TIMESTAMP)) AS dep_dow
  FROM v_flights_ord f
)
SELECT
  b.*,
  COALESCE(p.pax_total,0)        AS pax_total,
  COALESCE(p.lap_childs,0)       AS lap_childs,
  COALESCE(p.basic_econ,0)       AS basic_econ,
  COALESCE(p.stroller_users,0)   AS stroller_users,
  COALESCE(p.distinct_pnrs,0)    AS distinct_pnrs,
  COALESCE(bg.checked_bags,0)    AS checked_bags,
  COALESCE(bg.transfer_bags,0)   AS transfer_bags,
  CASE WHEN COALESCE(bg.checked_bags,0) > 0
       THEN CAST(bg.transfer_bags AS DOUBLE)/bg.checked_bags
       ELSE NULL END             AS transfer_ratio,
  COALESCE(s.ssr_count,0)        AS ssr_count,
  apt.iso_country_code           AS arrival_country
FROM base b
LEFT JOIN v_pnr_agg  p  USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN v_bags_agg bg USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN v_ssr_agg  s  USING (company_id, flight_number, scheduled_departure_date_local,
                               scheduled_departure_station_code, scheduled_arrival_station_code)
LEFT JOIN airports apt
       ON b.scheduled_arrival_station_code = apt.airport_iata_code
""").df()

print("Master rows:", len(df_master))
assert len(df_master) == n_flights, "Row-count mismatch: master != flights_ORD"


In [ ]:

df = df_master.copy()


if "dep_delay_min" not in df.columns:
    df["scheduled_departure_datetime_local"] = pd.to_datetime(df["scheduled_departure_datetime_local"], errors='coerce')
    df["actual_departure_datetime_local"]    = pd.to_datetime(df["actual_departure_datetime_local"], errors='coerce')
    df["dep_delay_min"] = (df["actual_departure_datetime_local"] - df["scheduled_departure_datetime_local"]).dt.total_seconds() / 60

df["is_difficult"] = (df["dep_delay_min"] > 15).astype(int)


df["load_factor"] = (df["pax_total"] / df["total_seats"])
if "slack_mins" not in df.columns:
    df["slack_mins"] = df["scheduled_ground_time_minutes"] - df["minimum_turn_minutes"]

checked  = df["checked_bags"].fillna(0)
transfer = df["transfer_bags"].fillna(0)
total_bags = checked + transfer
df["transfer_share"] = np.where(total_bags > 0, transfer / total_bags, 0.0)
df["bag_volume"]     = total_bags

pax = df["pax_total"].fillna(0)
df["ssr_per_100pax"] = np.where(pax > 0, df["ssr_count"] / pax * 100, 0.0)


if "dep_hour" not in df.columns:
    df["dep_hour"] = pd.to_datetime(df["scheduled_departure_datetime_local"], errors='coerce').dt.hour
mode_series = df["dep_hour"].mode()
df["dep_hour"] = df["dep_hour"].fillna(int(mode_series.iloc[0]) if len(mode_series) else 0).astype(int)
df["dep_hour_sin"] = np.sin(2 * np.pi * df["dep_hour"] / 24)
df["dep_hour_cos"] = np.cos(2 * np.pi * df["dep_hour"] / 24)


for col in ["fleet_type", "carrier", "arrival_country"]:
    df[col] = df[col].fillna("UNK").astype(str)


TRAIN_FEATURES_NUM = [
    "slack_mins", "load_factor", "ssr_count", "ssr_per_100pax",
    "transfer_share", "bag_volume", "dep_hour_sin", "dep_hour_cos",
    "dep_dow", "checked_bags", "transfer_bags"
]
TRAIN_FEATURES_CAT = ["fleet_type", "carrier", "arrival_country"]
TRAIN_FEATURES_ALL = TRAIN_FEATURES_NUM + TRAIN_FEATURES_CAT

model_df = df[["company_id","flight_number","scheduled_departure_date_local","dep_delay_min","is_difficult"] + TRAIN_FEATURES_ALL].copy()
model_df[TRAIN_FEATURES_NUM] = model_df[TRAIN_FEATURES_NUM].fillna(0)

print("Modeling frame shape:", model_df.shape)
print(f"Target distribution (% difficult): {model_df['is_difficult'].mean()*100:.1f}%")
display(model_df.head(3))

X = model_df.drop(columns=["company_id","flight_number","scheduled_departure_date_local","dep_delay_min","is_difficult"])
y = model_df["is_difficult"]


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

pre = ColumnTransformer(
    [
        ("num", StandardScaler(), TRAIN_FEATURES_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), TRAIN_FEATURES_CAT),
    ],
    remainder="drop"
)



logreg = LogisticRegression(
    solver="saga",
    max_iter=5000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

pipe = Pipeline([("pre", pre), ("clf", logreg)])
pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
prec, rec, thr = precision_recall_curve(y_test, proba)
f1_scores = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = thr[np.nanargmax(f1_scores)]

print("Model training complete!")
print(f"ROC AUC: {auc:.3f}")
print(f"Best threshold (F1): {best_thr:.3f}")

y_pred = (proba >= best_thr).astype(int)
print("\nClassification Report @best F1:")
print(classification_report(y_test, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipe_rf = Pipeline(steps=[("pre", pre), ("clf", rf)])
pipe_rf.fit(X_train, y_train)

rf_proba = pipe_rf.predict_proba(X_test)[:, 1]
rf_pred  = (rf_proba >= 0.5).astype(int)

rf_auc = roc_auc_score(y_test, rf_proba)
print("Random Forest ROC AUC:", round(rf_auc, 3))
print("\nClassification Report (0.5 threshold):\n", classification_report(y_test, rf_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))


ohe = pipe_rf.named_steps["pre"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(TRAIN_FEATURES_CAT)
feat_names = TRAIN_FEATURES_NUM + list(cat_feature_names)

importances = pipe_rf.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"Feature": feat_names, "Importance": importances}) \
           .sort_values("Importance", ascending=False).head(15)
print("\nTop 15 Feature Importances:")
display(imp_df)


In [ ]:

for c in ["load_factor","slack_mins","transfer_share","bag_volume","ssr_per_100pax",
          "dep_hour","dep_hour_sin","dep_hour_cos"]:
    df_master[c] = df[c]


X_score = df_master[TRAIN_FEATURES_ALL]
df_master["difficulty_score"] = pipe_rf.predict_proba(X_score)[:, 1]


df_master["daily_rank"] = (
    df_master.groupby("scheduled_departure_date_local")["difficulty_score"]
    .rank(method="first", ascending=False)
)


group_sizes = df_master.groupby("scheduled_departure_date_local")["difficulty_score"].transform("size")
df_master["rank_pct"] = ((df_master["daily_rank"] - 1) / (group_sizes - 1)).fillna(0).clip(0, 1)

df_master["difficulty_class"] = pd.cut(
    df_master["rank_pct"],
    bins=[0, 0.2, 0.5, 1.0],
    labels=["Difficult", "Medium", "Easy"],
    include_lowest=True
)

submission_cols = [
    "company_id","flight_number","scheduled_departure_date_local",
    "scheduled_departure_station_code","scheduled_arrival_station_code",

    *TRAIN_FEATURES_ALL,

    "difficulty_score","daily_rank","difficulty_class"
]

submission = (
    df_master[submission_cols]
    .sort_values(["scheduled_departure_date_local","daily_rank"])
    .reset_index(drop=True)
)

out_path = "test_querykings.csv"
submission.to_csv(out_path, index=False)
print(f"Submission file created: {out_path}, shape={submission.shape}")
display(submission.sample(10))


Part 3 Insights

In [ ]:

dest_difficulty = (
    df_master.groupby("scheduled_arrival_station_code")["difficulty_score"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(8,5), dpi=150)
sns.barplot(x=dest_difficulty.values, y=dest_difficulty.index, palette="Reds_r")
plt.title("Top 10 Difficult Destinations (Avg. Score)", fontsize=14, weight="bold")
plt.xlabel("Average Difficulty Score")
plt.ylabel("Destination")
plt.tight_layout()
plt.show()


In [ ]:

df_master["dom_int"] = df_master["arrival_country"].apply(lambda x: "Domestic" if x=="US" else "International")


In [ ]:

dom_int_scores = df_master.groupby("dom_int")["difficulty_score"].mean()

plt.figure(figsize=(6,5), dpi=150)
sns.barplot(x=dom_int_scores.index, y=dom_int_scores.values,
            palette=["skyblue","salmon"], edgecolor="black")

plt.title("Average Difficulty Score — Domestic vs International", fontsize=14, weight="bold")
plt.ylabel("Average Difficulty Score")
plt.xlabel("")
plt.ylim(0,1)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

dest_difficulty = (
    df_master.groupby("scheduled_arrival_station_code")["difficulty_score"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

top_dests = dest_difficulty.index.tolist()


drivers = (
    df_master.groupby("scheduled_arrival_station_code")[
        ["slack_mins","transfer_share","ssr_per_100pax","bag_volume","load_factor"]
    ]
    .mean()
    .fillna(0)
)


drivers_top = drivers.loc[top_dests]


drivers_norm = (drivers_top - drivers_top.min()) / (drivers_top.max() - drivers_top.min())


plt.figure(figsize=(9,6), dpi=150)
sns.heatmap(drivers_norm, annot=True, cmap="Greens", cbar_kws={"label":"Relative Intensity"}, linewidths=0.5)

plt.title("Operational Drivers for Top 10 Difficult Destinations", fontsize=14, weight="bold", pad=12)
plt.xlabel("Driver", fontsize=12, weight="bold")
plt.ylabel("Destination", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score

def evaluate_model(name, model, X_train, X_test, y_train, y_test):

    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:,1]
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_proba)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    return {"Model": name, "AUC": auc, "Recall": recall, "Precision": precision, "Accuracy": acc}


results = []
results.append(evaluate_model("Logistic Regression", pipe, X_train, X_test, y_train, y_test))
results.append(evaluate_model("Random Forest", pipe_rf, X_train, X_test, y_train, y_test))

import pandas as pd
results_df = pd.DataFrame(results)


results_melted = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(8,5), dpi=150)
sns.barplot(data=results_melted, x="Metric", y="Score", hue="Model", palette="Set2")
plt.title("Model Performance Comparison", fontsize=14, weight="bold")
plt.ylim(0,1)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

results_df


In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
from sklearn.ensemble import RandomForestClassifier




log_reg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
pipe_log = Pipeline([("pre", pre), ("clf", log_reg)])
pipe_log.fit(X_train, y_train)

log_proba = pipe_log.predict_proba(X_test)[:,1]
log_pred = (log_proba >= 0.5).astype(int)

log_results = {
    "Model": "Logistic Regression",
    "AUC": roc_auc_score(y_test, log_proba),
    "Recall (Difficult)": recall_score(y_test, log_pred),
    "Precision (Difficult)": precision_score(y_test, log_pred),
    "F1 (Difficult)": f1_score(y_test, log_pred)
}


rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
pipe_rf = Pipeline([("pre", pre), ("clf", rf)])
pipe_rf.fit(X_train, y_train)

rf_proba = pipe_rf.predict_proba(X_test)[:,1]
rf_pred = (rf_proba >= 0.5).astype(int)

rf_results = {
    "Model": "Random Forest",
    "AUC": roc_auc_score(y_test, rf_proba),
    "Recall (Difficult)": recall_score(y_test, rf_pred),
    "Precision (Difficult)": precision_score(y_test, rf_pred),
    "F1 (Difficult)": f1_score(y_test, rf_pred)
}


import pandas as pd
comparison = pd.DataFrame([log_results, rf_results])
print(comparison)


comparison_melted = comparison.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(8,5), dpi=150)
sns.barplot(data=comparison_melted, x="Metric", y="Score", hue="Model", palette="Set2")
plt.title("Model Comparison: Logistic vs Random Forest", fontsize=14, weight="bold")
plt.ylim(0,1)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


What if analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


difficult_flights = (
    df_master[df_master["difficulty_class"]=="Difficult"]
    .sort_values("difficulty_score", ascending=False)
    .head(5)
    .copy()
)


scenarios = np.arange(0, 61, 10)

results = []
for idx, row in difficult_flights.iterrows():
    for extra in scenarios:
        modified = row.copy()
        modified["slack_mins"] = row["slack_mins"] + extra

        X_sim = pd.DataFrame([modified[TRAIN_FEATURES_ALL]])
        score = pipe_rf.predict_proba(X_sim)[:,1][0]
        results.append({
            "flight_number": row["flight_number"],
            "extra_slack": extra,
            "difficulty_score": score
        })

sim_df = pd.DataFrame(results)


plt.figure(figsize=(8,5), dpi=150)
for flight in sim_df["flight_number"].unique():
    subset = sim_df[sim_df["flight_number"]==flight]
    plt.plot(subset["extra_slack"], subset["difficulty_score"], marker="o", label=f"Flight {flight}")

plt.axhline(0.5, color="gray", linestyle="--", alpha=0.7, label="50% Difficulty Threshold")
plt.title("What-If Analysis: Effect of Extra Ground Time on Difficulty", fontsize=14, weight="bold")
plt.xlabel("Additional Slack Minutes")
plt.ylabel("Predicted Difficulty Score")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

summary = (
    sim_df.groupby("extra_slack")["difficulty_score"]
    .mean()
    .reset_index()
    .rename(columns={"difficulty_score":"avg_score"})
)


baseline = summary.loc[summary["extra_slack"]==0, "avg_score"].values[0]
summary["reduction_vs_baseline"] = (baseline - summary["avg_score"]) / baseline * 100

print("Business Insight: Average effect of adding slack minutes")
print(summary)


In [ ]:
# --- Pick a few sample difficult flights again ---
difficult_flights_ssr = df_master[df_master["difficulty_class"]=="Difficult"].head(3).copy()


ssr_reductions = [0, 5, 10, 15]

results_ssr = []
for idx, row in difficult_flights_ssr.iterrows():
    for reduction in ssr_reductions:
        modified = row.copy()
        modified["ssr_count"] = max(0, row["ssr_count"] - reduction)
        X_sim = pd.DataFrame([modified[TRAIN_FEATURES_ALL]])
        score = pipe_rf.predict_proba(X_sim)[:,1][0]
        results_ssr.append({
            "flight_number": row["flight_number"],
            "ssr_reduction": reduction,
            "difficulty_score": score
        })

sim_ssr_df = pd.DataFrame(results_ssr)


summary_ssr = (
    sim_ssr_df.groupby("ssr_reduction")["difficulty_score"]
    .mean()
    .reset_index()
    .rename(columns={"difficulty_score":"avg_score"})
)

baseline_ssr = summary_ssr.loc[summary_ssr["ssr_reduction"]==0, "avg_score"].values[0]
summary_ssr["reduction_vs_baseline"] = (baseline_ssr - summary_ssr["avg_score"]) / baseline_ssr * 100

print("Business Insight: Average effect of reducing SSR count")
print(summary_ssr)

# --- Plot ---
plt.figure(figsize=(8,5), dpi=150)
plt.plot(summary_ssr["ssr_reduction"], summary_ssr["avg_score"], marker="o", color="navy")
plt.title("What-If Analysis: Effect of Reducing SSR Requests", fontsize=14, weight="bold")
plt.xlabel("SSR Reduction (number of requests)")
plt.ylabel("Predicted Difficulty Score (avg)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Flight difficulty heatmap

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier


sample_date = None
top_k_dest = 20
show_outline = True



if "dep_hour" not in df_master.columns:
    df_master["scheduled_departure_datetime_local"] = pd.to_datetime(df_master["scheduled_departure_datetime_local"])
    df_master["dep_hour"] = df_master["scheduled_departure_datetime_local"].dt.hour


if "slack_mins" not in df_master.columns:
    df_master["slack_mins"] = (
        df_master["scheduled_ground_time_minutes"] - df_master["minimum_turn_minutes"]
    )



if ("transfer_share" not in df_master.columns) or ("bag_volume" not in df_master.columns):
    checked = df_master["checked_bags"].fillna(0)
    transfer = df_master["transfer_bags"].fillna(0)
    total_bags = checked + transfer
    df_master["transfer_share"] = np.where(total_bags > 0, transfer / total_bags, 0.0)
    df_master["bag_volume"] = total_bags


if "ssr_per_100pax" not in df_master.columns:
    pax = df_master["pax_total"].fillna(0)
    df_master["ssr_per_100pax"] = np.where(pax > 0, df_master["ssr_count"] / pax * 100, 0.0)


if "dep_hour_sin" not in df_master.columns or "dep_hour_cos" not in df_master.columns:
    df_master["dep_hour_sin"] = np.sin(2 * np.pi * df_master["dep_hour"] / 24)
    df_master["dep_hour_cos"] = np.cos(2 * np.pi * df_master["dep_hour"] / 24)


if "dep_dow" not in df_master.columns:
    df_master["dep_dow"] = pd.to_datetime(df_master["scheduled_departure_datetime_local"]).dt.dayofweek


for c in ["fleet_type","carrier","arrival_country"]:
    if c in df_master.columns:
        df_master[c] = df_master[c].fillna("UNK").astype(str)


feature_cols = [
    "slack_mins","load_factor","ssr_count","ssr_per_100pax",
    "transfer_share","bag_volume","dep_hour_sin","dep_hour_cos","dep_dow",
    "checked_bags","transfer_bags","fleet_type","carrier","arrival_country"
]
cat_cols = ["fleet_type","carrier","arrival_country"]
num_cols = [c for c in feature_cols if c not in cat_cols]

missing_feats = [c for c in feature_cols if c not in df_master.columns]
if missing_feats:
    raise KeyError(f"Missing features on df_master: {missing_feats}")

X_all = df_master[feature_cols].copy()
X_all[num_cols] = X_all[num_cols].fillna(0)

need_train = False
if "pipe_rf" not in globals():
    need_train = True
else:
    try:
        _ = pipe_rf.predict_proba(X_all.head(1))
    except Exception:
        need_train = True

if need_train:
    pre_local = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ],
        remainder="drop"
    )
    rf_local = RandomForestClassifier(
        n_estimators=300, max_depth=None, class_weight="balanced",
        random_state=42, n_jobs=-1
    )
    pipe_rf = Pipeline([("pre", pre_local), ("clf", rf_local)])

    if "is_difficult" not in df_master.columns:
        if "dep_delay_min" not in df_master.columns:
            df_master["scheduled_departure_datetime_local"] = pd.to_datetime(df_master["scheduled_departure_datetime_local"])
            df_master["actual_departure_datetime_local"] = pd.to_datetime(df_master["actual_departure_datetime_local"])
            df_master["dep_delay_min"] = (
                (df_master["actual_departure_datetime_local"] - df_master["scheduled_departure_datetime_local"])
                .dt.total_seconds() / 60
            )
        df_master["is_difficult"] = (df_master["dep_delay_min"] > 15).astype(int)
    y_all = df_master["is_difficult"].astype(int)
    pipe_rf.fit(X_all, y_all)

if "difficulty_score" not in df_master.columns:
    df_master["difficulty_score"] = pipe_rf.predict_proba(X_all)[:, 1]

dates = df_master["scheduled_departure_date_local"].astype(str)
if sample_date is None or sample_date not in set(dates):
    sample_date = dates.value_counts().idxmax()  # busiest day

df_day = df_master[dates == sample_date].copy()
if df_day.empty:
    raise ValueError("No flights found for the selected day.")


top_dests = (
    df_day.groupby("scheduled_arrival_station_code")
    .size()
    .sort_values(ascending=False)
    .head(top_k_dest)
    .index
)

df_day = df_day[df_day["scheduled_arrival_station_code"].isin(top_dests)].copy()


all_hours = list(range(24))


pivot_mean = (
    df_day.pivot_table(
        index="scheduled_arrival_station_code",
        columns="dep_hour",
        values="difficulty_score",
        aggfunc="mean"
    )
    .reindex(columns=all_hours)
    .sort_index()
)


row_order = pivot_mean.mean(axis=1).sort_values(ascending=False).index
pivot_mean = pivot_mean.reindex(index=row_order)


plt.figure(figsize=(12, max(6, 0.35*len(pivot_mean))), dpi=150)
ax = sns.heatmap(
    pivot_mean,
    cmap="YlOrRd",
    vmin=0, vmax=1,
    linewidths=0.4 if show_outline else 0,
    linecolor="white" if show_outline else None,
    cbar_kws={"label": "Avg Difficulty Score"}
)

ax.set_title(f"Daily Flight Risk Heatmap — {sample_date}\nDestination × Departure Hour (Avg Difficulty Score)",
             fontsize=14, fontweight="bold", pad=10)
ax.set_xlabel("Departure Hour (Local)", fontsize=11, fontweight="bold")
ax.set_ylabel("Destination", fontsize=11, fontweight="bold")
ax.set_xticks(np.arange(0.5, 24.5, 1))
ax.set_xticklabels(all_hours, rotation=0)
ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.show()


Economical analysis

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


COST_PER_MIN_OPTIONS = [50, 75, 100]
REDUCTION_MINUTES = [3, 5, 10]
SHOW_COST_PER_MIN_FOR_DAILY = 75


df = df_master.copy()


if "dep_delay_min" not in df.columns:
    df["scheduled_departure_datetime_local"] = pd.to_datetime(df["scheduled_departure_datetime_local"])
    df["actual_departure_datetime_local"] = pd.to_datetime(df["actual_departure_datetime_local"])
    df["dep_delay_min"] = (
        (df["actual_departure_datetime_local"] - df["scheduled_departure_datetime_local"])
        .dt.total_seconds() / 60
    )

df["dep_delay_pos"] = df["dep_delay_min"].clip(lower=0)

if "difficulty_class" not in df.columns:
    if "difficulty_score" not in df.columns:
        raise KeyError("difficulty_score or difficulty_class not found. Run the scoring step first.")
    df["daily_rank"] = (
        df.groupby("scheduled_departure_date_local")["difficulty_score"]
          .rank(method="first", ascending=False)
    )
    df["rank_pct"] = (
        (df["daily_rank"] - 1) /
        (df.groupby("scheduled_departure_date_local")["difficulty_score"].transform("size") - 1)
    )
    df["difficulty_class"] = pd.cut(
        df["rank_pct"].clip(0,1),
        bins=[0,0.2,0.5,1.0],
        labels=["Difficult","Medium","Easy"],
        include_lowest=True
    )


class_delay = (
    df.groupby("difficulty_class")["dep_delay_pos"]
      .mean()
      .reindex(["Easy","Medium","Difficult"])
)


difficult = df[df["difficulty_class"] == "Difficult"].copy()
savings_matrix = []
for red in REDUCTION_MINUTES:
    mins_saved = np.minimum(difficult["dep_delay_pos"].values, red).sum()
    row = {"Reduction (min)": red, "Total minutes saved": mins_saved}
    for cpm in COST_PER_MIN_OPTIONS:
        row[f"${cpm}/min"] = mins_saved * cpm
    savings_matrix.append(row)
savings_df = pd.DataFrame(savings_matrix)
heatmap_df = savings_df.set_index("Reduction (min)")[[f"${c}/min" for c in COST_PER_MIN_OPTIONS]]


daily_cost = (
    df.groupby("scheduled_departure_date_local")["dep_delay_pos"]
      .sum()
      .sort_index()
      * SHOW_COST_PER_MIN_FOR_DAILY
)
daily_cost = daily_cost.rename(f"Total Cost @ ${SHOW_COST_PER_MIN_FOR_DAILY}/min")



colors = {"Easy":"#2ecc71","Medium":"#f39c12","Difficult":"#e74c3c"}


fig, ax = plt.subplots(figsize=(8,5), dpi=150)
bars = ax.bar(class_delay.index, class_delay.values,
              color=[colors[cat] for cat in class_delay.index],
              edgecolor='white', linewidth=2, alpha=0.9)
for bar, v in zip(bars, class_delay.values):
    ax.text(bar.get_x()+bar.get_width()/2., v+0.8, f"{v:.1f} min",
            ha='center', va='bottom', fontsize=11, weight='bold', color="#2c3e50")
ax.set_title("Average Departure Delay by Difficulty Class", fontsize=15, weight='bold', pad=15)
ax.set_ylabel("Average Delay (minutes)", fontsize=12, weight='bold')
ax.set_ylim(0, max(class_delay.values)*1.3)
plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(8,5.5), dpi=150)
sns.heatmap(heatmap_df/1000, annot=True, fmt='.1f',
            cmap='RdYlGn', linewidths=2, linecolor='white',
            cbar_kws={'label': 'Estimated Savings ($1000s)', 'shrink': 0.8},
            square=True, annot_kws={'size':11, 'weight':'bold'}, ax=ax)
ax.set_title("Potential Cost Savings: Reducing Delays on Difficult Flights",
             fontsize=15, weight='bold', pad=15)
ax.set_xlabel("Cost per Delay Minute", fontsize=12, weight='bold')
ax.set_ylabel("Avg Reduction per Difficult Flight (minutes)", fontsize=12, weight='bold')
plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(10,5), dpi=150)
daily_cost.plot(kind="bar", color="#34495e", edgecolor="white", linewidth=1.2, ax=ax)
for i, v in enumerate(daily_cost.values):
    ax.text(i, v+max(daily_cost.values)*0.01, f"${v/1000:.1f}K",
            ha='center', va='bottom', fontsize=10, weight='bold', color="#2c3e50")
ax.set_title(f"Daily Cost of Departure Delays @ ${SHOW_COST_PER_MIN_FOR_DAILY}/min",
             fontsize=15, weight='bold', pad=15)
ax.set_ylabel("Total Cost ($)", fontsize=12, weight='bold')
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


print("\nBaseline context (avg positive delay by class, minutes):")
print(class_delay.round(2))

print("\nSavings scenarios (Difficult flights only):")
print(savings_df.round(0))

print("\nDaily total cost @ chosen rate:")
print(daily_cost.round(0))
